## HW: outliers

**Name:** My Full Name  
**ID number:** 0123456789

## Import data and basic processing

The station and year are the same as in the previous homeworks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# write here the name of the .csv file you downloaded from the IMS
filename = "my_station.csv"

df = pd.read_csv(filename, na_values=['-'])

# the IMS column that holds the time starts with "Date & Time"
time_column = [c for c in df.columns if c.startswith('Date & Time')][0]
df['date'] = pd.to_datetime(df[time_column], dayfirst=True)
df = df.set_index('date')

# shorter names for the columns we will use
df = df.rename(columns={'Temperature (°C)': 'temperature',
                        'Rainfall (mm)': 'rain',
                        'Relative humidity (%)': 'humidity',
                        'Global radiation (W/m^2)': 'radiation'})
df.head()

Now choose 10 consecutive days with clear daily cycles, and check that there are no gaps (the number of missing values should be 0).

In [ ]:
# change the dates to a period of your choice, in the year of your data
start = '2024-01-05'
end = '2024-01-14'
temp_original = df.loc[start:end, 'temperature']

print("records:", len(temp_original), " (10 days with a record every 10 minutes should have 1440)")
print("missing values (NaN):", temp_original.isna().sum())

## The problems

The next cell adds problems to your temperature, in random places:

* 12 **spikes**, up or down, between 1 and 2.5 standard deviations of the series;
* 3 codes of **no data**, the value -9999;
* a **stuck sensor**: 40 points in a row with the same value.

You don't need to understand the cell, only run it. Change the `seed` to the last digits of your ID number. The series you must work with is `temp_bad`. The dataframe `truth` says where each problem is, and `temp_original` is the series before the problems. **Do not peek at them until a task tells you to.**

In [ ]:
def corrupt(series, seed):
    rng = np.random.default_rng(seed)
    n = len(series)
    bad = series.copy()
    truth = pd.DataFrame(False, index=series.index, columns=['spike', 'no_data', 'stuck'])
    # a sensor stuck for 40 points (a bit more than 6 hours)
    start = rng.integers(100, n - 200)
    bad.iloc[start:start + 40] = bad.iloc[start]
    truth.iloc[start:start + 40, 2] = True
    # 15 other random points, away from the stuck sensor: 12 spikes and 3 codes of "no data"
    free = np.setdiff1d(np.arange(50, n - 50), np.arange(start - 5, start + 46))
    picks = rng.choice(free, size=15, replace=False)
    spikes, no_data = picks[:12], picks[12:]
    bad.iloc[spikes] += rng.choice([-1, 1], size=12) * rng.uniform(1, 2.5, size=12) * series.std()
    truth.iloc[spikes, 0] = True
    bad.iloc[no_data] = -9999
    truth.iloc[no_data, 1] = True
    return bad, truth

seed = 123   # change it to the last digits of your ID number
temp_bad, truth = corrupt(temp_original, seed)

Two functions to help you. The first one, `conseq_series`, is from the challenge: it replaces with NaN the rows that have `N` or more equal values in a row. The second one, `score`, tells you how many spikes a method found, how many it missed, and how many false alarms it had. It uses `temp_clean`, that you will make in part 1.

In [ ]:
def conseq_series(series, N):
    # from the challenge: NaN where there are N or more equal values in a row
    sumsum_series = (series.diff() != 0).astype('int').cumsum()
    mask_outliers = series.groupby(sumsum_series).transform('count').ge(N)
    return pd.Series(np.where(mask_outliers, np.nan, series), index=series.index)

def score(found):
    """found = True where your method says there is an outlier"""
    spikes_found = (found & truth['spike']).sum()
    spikes_missed = (~found & truth['spike']).sum()
    false_alarms = (found & ~truth['spike'] & temp_clean.notna()).sum()
    return spikes_found, spikes_missed, false_alarms

## Part 1: common sense first

### The series with the problems

**My prediction** (sketch it on paper first, and describe your sketch here, BEFORE running the code or asking an AI):



In [ ]:
# code to process data and plot

**What I found, and why** (compare with your prediction, explain any surprise):



What is wrong with the graph? What did the code "no data" do to it?

### Common sense checks

In [ ]:
# code to process data and plot

**What I found, and why** (compare with your prediction, explain any surprise):



What did the -9999 do to the mean and the standard deviation? What would the Z-score method have found in `temp_bad`? How many problems of `truth` did your checks find?

## Part 2: global methods

**My prediction** (sketch it on paper first, and describe your sketch here, BEFORE running the code or asking an AI):



In [ ]:
# code to process data and plot

**What I found, and why** (compare with your prediction, explain any surprise):



Use `score`. Why do these methods fail in this series?

## Part 3: sliding windows

### The three functions

Save them also in a file, you will want to use them again.

In [ ]:
def rolling_zscore_outliers(series, window, k=3):
    # write your code here. It must return True where there is an outlier
    ...

def rolling_iqr_outliers(series, window, k=1.5):
    ...

def rolling_mad_outliers(series, window, k=3):
    ...

### Using them, with a window of 19 points

**My prediction** (sketch it on paper first, and describe your sketch here, BEFORE running the code or asking an AI):



In [ ]:
# code to process data and plot

**What I found, and why** (compare with your prediction, explain any surprise):



What are the false alarms? Did any method find the stuck sensor? Why not?

### What if... the window and the thresholds change?

**My prediction** (sketch it on paper first, and describe your sketch here, BEFORE running the code or asking an AI):



In [ ]:
# code to process data and plot

**What I found, and why** (compare with your prediction, explain any surprise):



What did you learn from the table? What happens with a very short window? And with a very long one? Is there a setting that finds all the spikes with no false alarms?

## Part 4: repair

**My prediction** (sketch it on paper first, and describe your sketch here, BEFORE running the code or asking an AI):



In [ ]:
# code to process data and plot

**What I found, and why** (compare with your prediction, explain any surprise):



Which error is the smallest and which is the largest? Why? Does the error at the stuck sensor depend on the time of the day in which it got stuck? What does this tell you about filling gaps?

## Part 5: think like in the exam

Answer in your own words, in a few sentences each. When asked to sketch, do it on paper and describe your sketch.

**1. Global thresholds.**



**2. The sliding MAD envelope.**



**3. The 10,000 gaussian numbers.**



**4. A spike, a sliding mean and a sliding median.**



**5. 10 years of hourly temperature.**



**6. What to do with an outlier.**



## Bonus

Optional. Start by writing the question that motivated your graph.

In [ ]:
# code to process data and plot

## AI declaration

Which AI tools did you use, and what did you use them for?